## Encoding the samples
We now save the output of the last layer for the `[CLS]` token of the tokenized sentences in our datasets.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

# load bert
bert = AutoModelForMaskedLM.from_pretrained('google-bert/bert-base-uncased',  
                                            output_hidden_states=True, 
                                            output_attentions=True)

# load the tokenizer for bert
bert_tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

In [ ]:
import torch

# we should try to use the GPU since this is a heavy task
# use CPU by default, so the program still runs if no GPU is available
dev = 'cpu'
if torch.cuda.is_available():
    # but, if we have a GPU, try to use it
    dev = 'cuda'

# put the model on GPU, hopefully
bert.to(dev)

In [3]:
import pandas as pd
import numpy as np

# load the dataset created with save_samples.ipynb
# edit this line to process each of the 4 datasets
df = pd.read_parquet('../datasets/books.parquet')

# collect embeddings here
embed = []

In [ ]:
for sent in df['txt'].tolist():
    # .to(dev), hopefully passes the tokenization task to the GPU as well
    tok = bert_tokenizer(sent, truncation=True, max_length=512, return_tensors='pt').to(dev)

    # don't calculate gradients for backpropagation
    # this saves some time when running the model
    with torch.no_grad():
        # bert itself was set on GPU already, if it was available
        res = bert(**tok)
    
    last_hidden = res.hidden_states[-1]
    # do this on the CPU to save GPU memory, though
    embed.append(last_hidden[:,0,:].squeeze().cpu())

# matrix with all output vectors put together
matrix = torch.stack(embed).cpu().numpy().astype(np.float32)
# save to disk
np.save('../embeds/books_embed.npy', matrix)